In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from helpers.helper_functions import *
import os

# Handling text 2 exercise
[Handling text exercisses ADApted drom ADA 2018 final exam]

The Sheldon Cooper we all know and love (OK, some of us might not know him, and some might not love him) from the TV series "The Big Bang Theory" has gotten into an argument with Leonard from the same TV show. Sheldon insists that he knows the show better than anyone, and keeps making various claims about the show, which neither of them know how to prove or disprove. The two of them have reached out to you ladies and gentlemen, as data scientists, to help them. You will be given the full script of the series, with information on the episode, the scene, the person saying each dialogue line, and the dialogue lines themselves.

Leonard has challenged several of Sheldon's claims about the show, and throughout this exam you will see some of those and you will get to prove or disprove them, but remember: sometimes, we can neither prove a claim, nor disprove it!

## Task A: Picking up the shovel

**Note: You will use the data you preprocess in this task in all the subsequent ones.**

Our friends' argument concerns the entire show. We have given you a file in the `data/` folder that contains the script of every single episode. New episodes are indicated by '>>', new scenes by '>', and the rest of the lines are dialogue lines. Some lines are said by multiple people (for example, lines indicated by 'All' or 'Together'); **you must discard these lines**, for the sake of simplicity. However, you do not need to do it for Q1 in this task -- you'll take care of it when you solve Q2.

**Q1**. Your first task is to extract all lines of dialogue in each scene and episode, creating a dataframe where each row has the episode and scene where a dialogue line was said, the character who said it, and the line itself. You do not need to extract the proper name of the episode (e.g. episode 1 can appear as "Series 01 Episode 01 - Pilot Episode", and doesn't need to appear as "Pilot Episode"). Then, answer the following question: In total, how many scenes are there in each season? We're not asking about unique scenes; the same location appearing in two episodes counts as two scenes. You can use a Pandas dataframe with a season column and a scene count column as the response.

**Note: The data refers to seasons as "series".**

In [2]:
script_df = pd.DataFrame(columns=['line', 'character', 'episode', 'scene'])
filename="data/all_scripts.txt"
with open(filename) as file:
    episode = None
    scene = None
    for line in file:
        line = line.removesuffix("\n")
        if line.startswith(">>"):
            episode = line.removeprefix(">> ") 
        elif line.startswith("> "):
            scene = line.removeprefix("> ")
        else:
            character = line.split(":")[0]
            cleaned_line = line.removeprefix(character + ": ")
            script_df.loc[len(script_df)] = cleaned_line, character, episode, scene
    
print(script_df.head())
    

                                                line     character  \
0  So if a photon is directed through a plane wit...       Sheldon   
1                         Agreed, what’s your point?       Leonard   
2  There’s no point, I just think it’s a good ide...       Sheldon   
3                                         Excuse me?       Leonard   
4                                           Hang on.  Receptionist   

                                episode                        scene  
0  Series 01 Episode 01 – Pilot Episode  A corridor at a sperm bank.  
1  Series 01 Episode 01 – Pilot Episode  A corridor at a sperm bank.  
2  Series 01 Episode 01 – Pilot Episode  A corridor at a sperm bank.  
3  Series 01 Episode 01 – Pilot Episode  A corridor at a sperm bank.  
4  Series 01 Episode 01 – Pilot Episode  A corridor at a sperm bank.  


**Q2**. Now, let's define two sets of characters: all the characters, and recurrent characters. Recurrent characters are those who appear in more than one episode. For the subsequent sections, you will need to have a list of recurrent characters. Assume that there are no two _named characters_ (i.e. characters who have actual names and aren't referred to generically as "little girl", "grumpy grandpa", etc.) with the same name, i.e. there are no two Sheldons, etc. Generate a list of recurrent characters who have more than 90 dialogue lines in total, and then take a look at the list you have. If you've done this correctly, you should have a list of 20 names. However, one of these is clearly not a recurrent character. Manually remove that one, and print out your list of recurrent characters. To remove that character, pay attention to the _named character_ assumption we gave you earlier on. **For all the subsequent questions, you must only keep the dialogue lines said by the recurrent characters in your list.**

In [3]:
aggr_unique = script_df.groupby(['character']).agg('nunique')
aggr = script_df.groupby(['character']).agg('count')
all_the_characters = aggr.index.values
recurrent_characters = aggr_unique[(aggr_unique['episode'] > 1) & (aggr['line'] > 90)].index.values
recurrent_characters = set(recurrent_characters)
recurrent_characters.remove('Man')
print(recurrent_characters)
script_df = script_df[script_df['character'].isin(recurrent_characters)]

{'Leonard', 'Bernadette', 'Raj', 'Stuart', 'Howard', 'Kripke', 'Bert', 'Beverley', 'Arthur', 'Zack', 'Sheldon', 'Mrs Wolowitz', 'Wil', 'Penny', 'Mrs Cooper', 'Amy', 'Emily', 'Leslie', 'Priya'}


## Task B: Read the scripts carefully

### Part 1: Don't put the shovel down just yet

**Q3**. From each dialogue line, replace punctuation marks (listed in the EXCLUDE_CHARS variable provided in `helpers/helper_functions.py`) with whitespaces, and lowercase all the text. **Do not remove any stopwords, leave them be for all the questions in this task.**

In [4]:
from helpers.helper_functions import EXCLUDE_CHARS
import re
pattern = "[" + "".join(map(re.escape, EXCLUDE_CHARS)) + "]"
script_df['line'] = script_df['line'].str.replace(pattern, " ", regex=True).apply(str.lower)
script_df.head(2)

,line,character,episode,scene
0,so if a photon is directed through a plane wit...,Sheldon,Series 01 Episode 01 – Pilot Episode,A corridor at a sperm bank.
1,agreed what s your point,Leonard,Series 01 Episode 01 – Pilot Episode,A corridor at a sperm bank.


**Q4**. For each term, calculate its "corpus frequency", i.e. its number of occurrences in the entire series. Visualize the distribution of corpus frequency using a histogram. Explain your observations. What are the appropriate x and y scales for this plot?

In [17]:
script_df['terms'] = script_df['line'].str.split()
terms_df = pd.DataFrame(index=script_df['terms'].explode().unique())
all_terms = script_df['terms'].explode()
terms_df['count'] = all_terms.value_counts()
terms_df

,count
so,3187.0
if,2490.0
a,13518.0
photon,4.0
is,5444.0
...,...
skyped,1.0
corresponded,1.0
nickels,1.0
highlighted,1.0


### Part 2: Talkativity
**Q5**. For each of the recurrent characters, calculate their total number of words uttered across all episodes. Based on this, who seems to be the most talkative character?

In [21]:
# your code goes here
script_df['n_terms'] = script_df['terms'].apply(len)
talk_agg = script_df.groupby('character')['n_terms'].sum()
talk_agg.sort_values(ascending=False)

character
Sheldon         185388
Leonard         102496
Penny            79270
Howard           69505
Raj              60099
Amy              39933
Bernadette       27726
Stuart            7955
Mrs Cooper        3389
Beverley          2029
Priya             1940
Wil               1678
Emily             1571
Mrs Wolowitz      1459
Arthur            1451
Zack              1427
Leslie            1249
Kripke            1246
Bert              1146
Name: n_terms, dtype: int64

## Task D: The Detective's Hat

Sheldon claims that given a dialogue line, he can, with an accuracy of above 70%, say whether it's by himself or by someone else. Leonard contests this claim, since he believes that this claimed accuracy is too high.

**Q6**. Divide the set of all dialogue lines into two subsets: the training set, consisting of all the seasons except the last two, and the test set, consisting of the last two seasons.

In [24]:
test_seasons = ("Series 09", "Series 10")
train_df = script_df[~script_df['episode'].str.startswith(test_seasons)]
test_df = script_df[script_df['episode'].str.startswith(test_seasons)]

**Q7**. Find the set of all words in the training set that are only uttered by Sheldon. Is it possible for Sheldon to identify himself only based on these? Use the test set to assess this possibility, and explain your method.

In [43]:
char_agg = train_df.groupby('character')['terms'].sum()


In [46]:
char_agg_unique = char_agg.apply(set)
sheldon_terms = char_agg_unique.loc['Sheldon']
for char in char_agg_unique.index.unique():
    if char != "Sheldon":
        sheldon_terms = sheldon_terms - char_agg_unique.loc[char]


In [57]:
test_df['predicted_char_is_sheldon'] = test_df['terms'].apply(
    lambda terms: any(t in sheldon_terms for t in terms)
)
correct_predictions = (test_df['character'] == "Sheldon") & (test_df['predicted_char_is_sheldon'])
print(correct_predictions.mean())

0.032156554233454614


/var/folders/fc/sbtv4zc9113_kb2r0m_b6wx80000gn/T/ipykernel_3507/3072275481.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['predicted_char_is_sheldon'] = test_df['terms'].apply(
